# FAISS Index Builder — Example (No-IHC) Ablation

Builds a single FAISS index from `chunks_example_noihc.csv` (IHC training examples excluded).
Used for the ablation study in Section 4.1.

**Output:**
```
corpus/index/
  vdb_example_noihc.faiss  (~47k vectors)
  lookup_example_noihc.json
```

In [1]:
import json
from pathlib import Path
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModel
import sys

sys.path.insert(0, str(Path("..").resolve()))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CORPUS_DIR = Path("../../corpus")
INDEX_DIR  = CORPUS_DIR / "index"
CHUNKS_DIR = CORPUS_DIR / "chunks"

Using device: cuda


In [2]:
# Load example chunks with IHC excluded
df_noihc = pd.read_csv(CHUNKS_DIR / "chunks_example_noihc.csv")[["chunk_id", "text"]]

print(f"Example (no-IHC) chunks: {len(df_noihc):,}")

Example (no-IHC) chunks: 49,049


In [3]:
from retriever import encode

SBERT_HF_ID = "sentence-transformers/all-mpnet-base-v2"
print(f"Loading {SBERT_HF_ID} ...")
sbert_tokenizer = AutoTokenizer.from_pretrained(SBERT_HF_ID)
sbert_model     = AutoModel.from_pretrained(SBERT_HF_ID).eval().to(device)
embed_dim       = sbert_model.config.hidden_size
print(f"Embedding dim: {embed_dim}  |  device: {device}")
print()

texts     = df_noihc["text"].tolist()
chunk_ids = df_noihc["chunk_id"].to_numpy(dtype="int64")

vectors = encode(texts, sbert_model, sbert_tokenizer,
                 batch_size=64, max_length=128, use_mean_pool=True)
faiss.normalize_L2(vectors)

inner = faiss.IndexFlatIP(embed_dim)
index = faiss.IndexIDMap(inner)
index.add_with_ids(vectors, chunk_ids)

out_path = INDEX_DIR / "vdb_example_noihc.faiss"
faiss.write_index(index, str(out_path))
print(f"Saved {index.ntotal:,} vectors to {out_path.name}")

del sbert_model
if device.type == "cuda":
    torch.cuda.empty_cache()

Loading sentence-transformers/all-mpnet-base-v2 ...


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Encoding:  99%|█████████▉| 759/767 [00:36<00:00, 21.74it/s]

Encoding:  99%|█████████▉| 762/767 [00:36<00:00, 20.67it/s]

Encoding: 100%|█████████▉| 765/767 [00:36<00:00, 20.48it/s]

Encoding: 100%|██████████| 767/767 [00:36<00:00, 20.89it/s]

Saved 49,049 vectors to vdb_example_noihc.faiss


In [4]:
lookup = {str(cid): text for cid, text in zip(df_noihc["chunk_id"], df_noihc["text"])}
lookup_path = INDEX_DIR / "lookup_example_noihc.json"
with open(lookup_path, "w") as f:
    json.dump(lookup, f, ensure_ascii=False, indent=2)
print(f"Saved lookup_example_noihc.json  ({len(lookup):,} entries)")

Saved lookup_example_noihc.json  (49,049 entries)
